# Fabula — Unsloth QLoRA Fine-Tuning & GGUF Export (Task D-02)

This notebook fine-tunes `Qwen/Qwen2.5-3B-Instruct` in 4-bit on the Fabula 4-stage SFT mystery dataset (`train.jsonl`, 2,048 samples) using **Unsloth** on Kaggle's free Tesla T4 GPU.

### Pipeline Overview:
1. **Install Unsloth & Dependencies**
2. **Load 4-bit Base Model** (`unsloth/Qwen2.5-3B-Instruct-bnb-4bit`)
3. **Attach LoRA Adapters** ($r=16, \alpha=16$)
4. **Train for 2 Epochs** (~35-45 minutes)
5. **Save Adapters & Export to GGUF (Q4_K_M)** for local CPU game execution.

In [ ]:
# Step 1: Install Unsloth and dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
print("Unsloth installed successfully!")

In [ ]:
# Step 2: Load Model & Tokenizer in 4-bit
import torch
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float16,
    load_in_4bit=True,
)

# Step 3: Attach LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("Model loaded and LoRA adapters attached!")

In [ ]:
# Step 4: Load & Format Datasets
import json
from datasets import Dataset

TRAIN_FILE = "/kaggle/working/train.jsonl"
VAL_FILE = "/kaggle/working/val.jsonl"

def load_jsonl_messages(filepath):
    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                formatted_text = tokenizer.apply_chat_template(
                    item["messages"],
                    tokenize=False,
                    add_generation_prompt=False
                )
                data.append({"text": formatted_text})
    return Dataset.from_list(data)

print("Formatting datasets into ChatML format...")
train_dataset = load_jsonl_messages(TRAIN_FILE)
val_dataset = load_jsonl_messages(VAL_FILE)
print(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")

In [ ]:
# Step 5: Trainer Setup & Execution
from trl import SFTTrainer
from transformers import TrainingArguments

OUTPUT_DIR = "/kaggle/working/fabula_model_output"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=15,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir=OUTPUT_DIR,
        report_to="none",
    ),
)

print("Starting QLoRA Training...")
stats = trainer.train()
print(f"Training Finished in {round(stats.metrics['train_runtime']/60, 1)} minutes!")

In [ ]:
# Step 6: Save LoRA Adapters and Export GGUF Q4_K_M
ADAPTER_DIR = "/kaggle/working/fabula_qlora_adapter"
GGUF_DIR = "/kaggle/working/fabula_gguf"

print(f"Saving LoRA adapter to {ADAPTER_DIR}...")
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Exporting merged GGUF (Q4_K_M) to {GGUF_DIR}...")
model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m"
)
print("GGUF Export Complete! Download fabula_gguf from the output tab.")